In [12]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer

PROJECT_ROOT = Path.cwd().parent.parent
file_path = PROJECT_ROOT/"data"/"bonprix_sample_data.xlsx" 
df = pd.read_excel(file_path)
df.columns = [col.strip() for col in df.columns]
df.rename(columns = {'Product_Category': 'Department'}, inplace = True)

In [13]:
# 1. Build the product catalog using Otto columns

catalog_cols = [
    "Product_ID", "Product_Name", "Department", "Product_Sub_Category",
    "Gender_Target", "Brand_Name", "Product_Type", "Color", "Material", "Style", 
    "Season", "Occasion_Tag", "Product_Price", "Cost"
]

catalog = (
    df.dropna(subset=["Product_ID"])
    .sort_values("Event_Date")
    .drop_duplicates(subset="Product_ID", keep="last")[catalog_cols]
    .reset_index(drop=True)
)

# 2. Weight actions
ACTION_WEIGHTS = {
    "Bought": 5, "Wishlisted": 2, "Added_to_Cart": 3, "Reviewed": 2, 
    "Browsed": 1, "Exchanged": 1, "Returned": -2, "Support_Contact": 0
}
events = df.dropna(subset=["Customer_ID", "Product_ID"]).copy()
events["weight"] = events["Action"].map(ACTION_WEIGHTS).fillna(0)

# Adjust weights with ratings
has_rating = events["Rating"].notna()
events.loc[has_rating, "weight"] += (events.loc[has_rating, "Rating"] - 3)

# 3. Aggregate customer interactions
interactions = (
    events.groupby(["Customer_ID", "Product_ID"], as_index=False)
    .agg(
        score=("weight", "sum"),
        n_events=("Action", "count"),
        last_event_date=("Event_Date", "max"),
        actions=("Action", lambda a: sorted(set(a.dropna()))),
    )
)

# 4. Build Content Feature Matrix using Otto features
CONTENT_CATEGORICAL = [
    "Department", "Product_Sub_Category",
    "Gender_Target", "Brand_Name", "Product_Type", "Color",
    "Material", "Style", "Season", "Occasion_Tag"
]
CONTENT_NUMERIC = ["Product_Price"]

catalog_features = catalog.set_index("Product_ID")

# Fill missing values
for col in CONTENT_CATEGORICAL:
    catalog_features[col] = catalog_features[col].fillna("Unknown")
catalog_features[CONTENT_NUMERIC] = catalog_features[CONTENT_NUMERIC].fillna(0.0)

content_transformer = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CONTENT_CATEGORICAL),
        ("num", MinMaxScaler(), CONTENT_NUMERIC),
    ]
)

content_matrix = content_transformer.fit_transform(catalog_features)
content_matrix = np.asarray(content_matrix.todense()) if hasattr(content_matrix, "todense") else content_matrix
product_ids = catalog_features.index.to_numpy()
product_id_to_idx = {pid: i for i, pid in enumerate(product_ids)}

# 5. Package and save to Joblib
trained_model_artifacts = {
    "content_transformer": content_transformer,
    "interactions": interactions,
    "content_matrix": content_matrix,
    "product_id_to_idx": product_id_to_idx,
}

# Create model directory and save
output_path = "../models/bonprix_recommender_model.joblib"
joblib.dump(trained_model_artifacts, output_path)

print(f"Bonprix Model successfully exported to {output_path}!")

Bonprix Model successfully exported to ../models/bonprix_recommender_model.joblib!
